# 04 - Model Explainability

This notebook analyzes model predictions using explainability techniques.

In [ ]:
# =============================================================================
# Import Required Libraries for Model Explainability
# =============================================================================
# Data manipulation
import pandas as pd
import numpy as np

# SHAP: SHapley Additive exPlanations for model interpretation
import shap

# Visualization
import matplotlib.pyplot as plt

# Scikit-learn: model building and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# LightGBM: gradient boosting for SHAP TreeExplainer
import lightgbm as lgb

## Load Model and Data

In [ ]:
# =============================================================================
# Load Final Modeling Dataset
# =============================================================================
# Load the leakage-free prefix-based dataset from modeling notebook
# This ensures explainability uses EXACTLY the same features as the model

import pandas as pd

model_df = pd.read_csv("../data/processed/modeling_dataset_optionB_FINAL.csv")

# Separate target from features
y = model_df["target"]
X = model_df.drop(columns=["target"])

print("Explainability feature shape:", X.shape)
print("First 10 features:", X.columns[:10].tolist())

In [ ]:
# =============================================================================
# Define Feature Groups for Preprocessing
# =============================================================================
# Numeric features: Price statistics and variety counts
# - Price stats capture browsing behavior (mean, std, min, max)
# - Variety counts show exploration depth (categories, colors, models)

num_features = [
    'price_mean_pfx', 'price_std_pfx', 'price_min_pfx', 'price_max_pfx',
    'price2_mean_pfx', 'price2_std_pfx',
    'country_nunique_pfx',
    'page_1_(main_category)_nunique_pfx',
    'page_2_(clothing_model)_nunique_pfx',
    'colour_nunique_pfx',
    'model_photography_nunique_pfx'
]

# Categorical features: Mode (most common value) of categorical attributes
# These capture the user's dominant preferences in early clicks
cat_features = [
    'country_mode_pfx',
    'page_1_(main_category)_mode_pfx',
    'page_2_(clothing_model)_mode_pfx',
    'colour_mode_pfx',
    'model_photography_mode_pfx'
]

## Feature Importance

In [ ]:
# =============================================================================
# Train Logistic Regression for Coefficient Analysis
# =============================================================================
# Logistic Regression coefficients provide interpretable feature importance
# Positive coefficients → increase purchase probability
# Negative coefficients → decrease purchase probability

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),                                    # Standardize for comparable coefficients
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),  # One-hot encode categories
    ]
)

log_reg = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=3000))  # Increased iterations for convergence
])

log_reg.fit(X, y)
print("Logistic Regression trained for coefficient analysis")

## SHAP Analysis

In [ ]:
# =============================================================================
# Extract and Rank Logistic Regression Coefficients
# =============================================================================
# Build complete feature name list (numeric + one-hot encoded categorical)
# Sort by absolute coefficient value to identify most influential features

feature_names = (
    num_features +
    list(
        log_reg.named_steps["preprocess"]
        .named_transformers_["cat"]
        .get_feature_names_out(cat_features)
    )
)

# Extract coefficients from trained model
coefficients = log_reg.named_steps["clf"].coef_[0]

# Create ranked dataframe
coef_df = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients)  # For ranking by importance
    })
    .sort_values("abs_coefficient", ascending=False)
)

print("Top 10 Most Influential Features (Logistic Regression):")
coef_df.head(10)

In [ ]:
# =============================================================================
# Visualize Top Logistic Regression Coefficients
# =============================================================================
# Horizontal bar chart showing coefficient direction and magnitude
# Positive = increases purchase probability, Negative = decreases

top_n = 15

plt.figure(figsize=(8, 6))
plt.barh(
    coef_df.head(top_n)["feature"][::-1],       # Reverse for top-to-bottom
    coef_df.head(top_n)["coefficient"][::-1]
)
plt.title("Top Logistic Regression Drivers of Purchase Intent (Prefix-Based)")
plt.xlabel("Coefficient Value (Standardized)")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Train LightGBM for SHAP Analysis
# =============================================================================
# SHAP TreeExplainer works efficiently with tree-based models like LightGBM
# Must label-encode categorical features (LightGBM requires numeric input)

from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder

# Copy X to avoid modifying original data
X_lgbm = X.copy()

# Define categorical columns (same as before)
cat_features = [
    'country_mode_pfx',
    'page_1_(main_category)_mode_pfx',
    'page_2_(clothing_model)_mode_pfx',
    'colour_mode_pfx',
    'model_photography_mode_pfx'
]

# Label encode each categorical column
label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    X_lgbm[col] = le.fit_transform(X_lgbm[col].astype(str))
    label_encoders[col] = le  # Store for potential inverse transform

# Train LightGBM with regularization (same hyperparameters as modeling notebook)
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm_model.fit(X_lgbm, y)
print("LightGBM trained for SHAP analysis")

In [ ]:
# =============================================================================
# Compute SHAP Values Using TreeExplainer
# =============================================================================
# SHAP (SHapley Additive exPlanations) computes feature contributions
# TreeExplainer is optimized for tree-based models (exact computation)

import shap

explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_lgbm)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base prediction): {explainer.expected_value:.4f}")

In [ ]:
# =============================================================================
# Global Feature Importance (Bar Plot)
# =============================================================================
# Shows mean absolute SHAP value per feature across all samples
# Higher bars = more influential features for purchase prediction

shap.summary_plot(
    shap_values,
    X_lgbm,
    plot_type="bar",
    max_display=15,
    show=False
)
plt.title("Global Feature Importance for Purchase Intent (LightGBM, Prefix-Based Features)")
plt.tight_layout()
plt.show()

## Individual Prediction Explanation

In [ ]:
# =============================================================================
# SHAP Beeswarm Plot (Feature Impact Distribution)
# =============================================================================
# Each dot = one sample
# X-axis = SHAP value (impact on prediction)
# Color = feature value (red = high, blue = low)
# Shows how feature values affect predictions

shap.summary_plot(
    shap_values,
    X_lgbm,
    max_display=15,
    show=False
)
plt.title("Feature-Level SHAP Impact on Purchase Intent Prediction")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Cumulative Feature Importance
# =============================================================================
# Shows how many features are needed to explain 80% of model predictions
# Useful for feature selection and model simplification

shap_importance = np.abs(shap_values).mean(axis=0)      # Mean absolute SHAP per feature
sorted_vals = np.sort(shap_importance)[::-1]             # Sort descending
cum_vals = np.cumsum(sorted_vals) / sorted_vals.sum()    # Cumulative proportion

plt.figure(figsize=(8, 5))
plt.plot(cum_vals, linewidth=2)
plt.axhline(0.8, linestyle="--", color="red", label="80% threshold")
plt.xlabel("Number of Features (ranked by importance)")
plt.ylabel("Cumulative SHAP Importance")
plt.title("Cumulative Feature Contribution to Purchase Intent Prediction")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SHAP Dependence Plot
# =============================================================================
# Shows how a single feature's value affects SHAP contribution
# Automatically selects an interaction feature for coloring
# X-axis = feature value, Y-axis = SHAP impact

shap.dependence_plot(
    "page_2_(clothing_model)_nunique_pfx",  # Number of unique clothing models viewed
    shap_values,
    X,
    title="SHAP Dependence: Clothing Model Variety"
)

In [ ]:
# =============================================================================
# Force Plot: Individual Prediction Explanation
# =============================================================================
# Shows how each feature pushes prediction from base value
# Red = pushes toward purchase, Blue = pushes away from purchase
# Useful for explaining individual session predictions

idx = 0  # Example session (first row)
shap.force_plot(
    explainer.expected_value,  # Base prediction (average)
    shap_values[idx],          # SHAP values for this sample
    X.iloc[idx],               # Feature values for this sample
    matplotlib=True
)

In [ ]:
# =============================================================================
# Waterfall Plot: Detailed Feature Breakdown
# =============================================================================
# Shows cumulative feature contributions for a single prediction
# Starting from base value, each feature adds/subtracts to reach final output
# More detailed view than force plot

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx],           # SHAP values for sample
        base_values=explainer.expected_value,  # Starting point
        data=X.iloc[idx],                   # Feature values
        feature_names=X.columns.tolist()    # Column names
    )
)

In [ ]:
# =============================================================================
# Predicted Probability Distribution by Class
# =============================================================================
# Shows model calibration and class separation
# Good separation = model distinguishes purchase intent from browsing
# Overlapping distributions = harder to classify

import seaborn as sns

# Generate purchase probability predictions
p_lgbm = lgbm_model.predict_proba(X_lgbm)[:, 1]

# Plot density for each class
plt.figure(figsize=(8, 5))
sns.kdeplot(p_lgbm[y == 1], label="Quick Decision (Purchase Intent)", fill=True, alpha=0.5)
sns.kdeplot(p_lgbm[y == 0], label="Extended Browsing", fill=True, alpha=0.5)
plt.title("Predicted Purchase Intent Probability Distribution by Actual Outcome")
plt.xlabel("Predicted Probability of Quick Purchase Decision")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## SHAP Results

SHAP analysis reveals that early purchase intent is primarily driven by session-level behavioral features, particularly the number of product views. Price consistency also plays a significant role, with sessions exhibiting low price variability showing higher purchase likelihood. In contrast, extensive price comparison behavior negatively impacts early intent. Product categories contribute moderately, while visual attributes such as color and geographic information have relatively minor influence. Overall, the model demonstrates stable and interpretable behavior aligned with established e-commerce conversion patterns. Explainability was performed using SHAP for the LightGBM model and coefficient analysis for Logistic Regression. For SHAP analysis, the LightGBM model was retrained on the full modeling dataset using the same hyperparameters to ensure stable global feature attributions. Because the dataset does not contain an explicit purchase event, the target variable represents session-level purchase intent inferred from session completion. Accordingly, model explanations describe factors associated with early purchase-oriented behavior, not confirmed transactions.